# Mexico (MXN) — Fixed Income Derivatives

TIIE 28D swaps, CETES discount bills, MBONO fixed-rate bonds, and Udibonos (UDI-linked inflation bonds).

**Key features:**
- TIIE: unique 28-day periods (not quarterly or monthly)
- CETES: zero-coupon discount bills (ACT/360, MXN 10 face)
- MBONO: fixed-rate sovereign, semi-annual, ACT/360
- Udibono: UDI-denominated (daily inflation unit, Banxico)
- All instruments use ACT/360 day count

In [ ]:
import sys, os, math
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta

from pricebook.fixed_income.mexican import (
    TIIESwap, CETESBill, UDIBond,
    build_tiie_curve, synthetic_tiie_strip, synthetic_cetes_quotes,
)
from pricebook.fixed_income.sovereign_bonds import create_sovereign_bond
from pricebook.fixed_income.inflation_unit import get_inflation_unit, InflationUnitBond
from pricebook.viz import configure_theme, greeks_profile

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

REF = date(2024, 11, 4)
print(f"Reference date: {REF}")
print(f"TIIE 28D policy rate: ~10.50% (Banxico)")
print(f"UDI value: ~8.20 MXN")

## 1. TIIE Curve Construction

Build the TIIE discount curve from a synthetic swap strip. Mexico uses 28-day periods — the swap fixed leg pays every 28 days, not monthly or quarterly.

In [ ]:
# Synthetic TIIE swap strip (realistic Nov 2024)
strip = synthetic_tiie_strip(REF, tiie_28d=0.1050)
tiie_curve = build_tiie_curve(REF, strip)

print(f"TIIE Swap Strip ({len(strip)} points):")
print(f"{'Tenor':>8}  {'Maturity':>12}  {'Rate':>8}  {'DF':>10}")
print(f"{'─'*8}  {'─'*12}  {'─'*8}  {'─'*10}")
for c in strip:
    df = tiie_curve.df(c["maturity"])
    print(f"{c['tenor_months']:>6}M  {c['maturity']}  {c['rate']*100:>7.2f}%  {df:>10.6f}")

In [ ]:
# TIIE discount curve plot
from pricebook.viz._backend import apply_theme, create_figure

tenors_y = [c["years"] for c in strip]
rates_pct = [c["rate"] * 100 for c in strip]
dfs = [tiie_curve.df(c["maturity"]) for c in strip]

with apply_theme():
    fig, (ax1, ax2) = create_figure(2)

    ax1.plot(tenors_y, rates_pct, 'o-', linewidth=2, markersize=6)
    ax1.set_xlabel("Tenor (years)")
    ax1.set_ylabel("TIIE Rate (%)")
    ax1.set_title("TIIE 28D Swap Curve — Nov 2024")
    ax1.axhline(10.50, color='gray', linestyle='--', alpha=0.5, label='Policy rate 10.50%')
    ax1.legend()

    ax2.plot(tenors_y, dfs, 's-', linewidth=2, markersize=6, color='#059669')
    ax2.set_xlabel("Tenor (years)")
    ax2.set_ylabel("Discount Factor")
    ax2.set_title("TIIE Discount Factors")

    fig.tight_layout()

## 2. TIIE 28D Swap

The TIIE swap is Mexico's plain vanilla interest rate swap. The fixed leg pays every 28 days (not monthly). The floating leg references the 28-day TIIE fixing.

In [ ]:
# Price TIIE swaps at various tenors
tenors = [1, 2, 3, 5, 7, 10]
print(f"TIIE 28D Swap Pricing (MXN 1bn notional):")
print(f"{'Tenor':>6}  {'Fixed Rate':>10}  {'Par Rate':>10}  {'DV01 (MXN)':>12}  {'28D Periods':>12}")
print(f"{'─'*6}  {'─'*10}  {'─'*10}  {'─'*12}  {'─'*12}")

for t in tenors:
    swap = TIIESwap(REF, REF + relativedelta(years=t), fixed_rate=0.1050)
    r = swap.price(tiie_curve)
    print(f"{t:>4}Y  {r.fixed_rate*100:>9.2f}%  {r.par_rate*100:>9.2f}%  "
          f"{r.dv01:>12,.0f}  {r.n_periods:>12}")

## 3. CETES — Mexican T-Bills

Zero-coupon discount bills. Face value MXN 10. Quoted as discount rate. Standard tenors: 28D, 91D, 182D, 364D.

In [ ]:
# CETES pricing
quotes = synthetic_cetes_quotes(REF)
print(f"CETES Pricing (face = MXN 10):")
print(f"{'Days':>8}  {'Maturity':>12}  {'Discount':>10}  {'Price':>8}  {'Eq Yield':>10}")
print(f"{'─'*8}  {'─'*12}  {'─'*10}  {'─'*8}  {'─'*10}")

for q in quotes:
    cetes = CETESBill(q["maturity"], q["rate"])
    r = cetes.price(REF)
    print(f"{q['days']:>6}D  {q['maturity']}  {r.discount_rate*100:>9.2f}%  "
          f"{r.price:>8.4f}  {r.yield_rate*100:>9.2f}%")

## 4. MBONO — Fixed Rate Sovereign Bond

Semi-annual coupon, ACT/360. Priced via TIIE curve.

In [ ]:
# MBONO pricing at various tenors
coupons = [0.075, 0.0825, 0.10]
tenors = [3, 5, 10, 20, 30]

print(f"MBONO Pricing via TIIE Curve:")
print(f"{'Coupon':>8}  {'Tenor':>6}  {'Dirty Price':>12}")
print(f"{'─'*8}  {'─'*6}  {'─'*12}")

for cpn in coupons:
    for t in tenors:
        mat = REF + relativedelta(years=t)
        mbono = create_sovereign_bond("MBONO", REF, mat, cpn)
        price = mbono.dirty_price(tiie_curve)
        above_par = "↑" if price > 100 else "↓" if price < 100 else "="
        print(f"{cpn*100:>7.2f}%  {t:>4}Y  {price:>11.4f} {above_par}")
    print()

## 5. Udibono — UDI-Linked Inflation Bond

Bonds denominated in UDI (Unidades de Inversión). Real coupons; nominal value = real value × UDI. Priced using both the country-specific `UDIBond` class and the generic `InflationUnitBond` framework.

In [ ]:
# UDI-linked bond pricing — country-specific class
from pricebook.core.discount_curve import DiscountCurve
from pricebook.core.interpolation import InterpolationMethod

# Build a real (UDI) curve at ~4% real rate
udi_dates = [REF + relativedelta(years=y) for y in [1, 2, 3, 5, 10, 20, 30]]
udi_dfs = [math.exp(-0.04 * y) for y in [1, 2, 3, 5, 10, 20, 30]]
udi_curve = DiscountCurve(REF, udi_dates, udi_dfs,
                           interpolation=InterpolationMethod.LOG_LINEAR)

current_udi = 8.2046  # Nov 2024 approximate

print(f"Udibono Pricing (UDI = {current_udi:.4f} MXN):")
print(f"{'Tenor':>6}  {'Real Cpn':>10}  {'Real PV':>10}  {'Nominal PV':>12}  {'Real Yield':>10}")
print(f"{'─'*6}  {'─'*10}  {'─'*10}  {'─'*12}  {'─'*10}")

for t, cpn in [(3, 0.035), (5, 0.04), (10, 0.04), (20, 0.045), (30, 0.05)]:
    udi_bond = UDIBond(REF, REF + relativedelta(years=t), cpn, udi_at_issue=7.5)
    r = udi_bond.price(REF, udi_curve, current_udi)
    print(f"{t:>4}Y  {cpn*100:>9.2f}%  {r.real_price:>10.4f}  "
          f"{r.nominal_price:>11.2f}  {r.real_yield*100:>9.2f}%")

In [ ]:
# Breakeven inflation: nominal TIIE vs real UDI curves
from pricebook.fixed_income.inflation_unit import dual_curve_breakeven

bei = dual_curve_breakeven(tiie_curve, udi_curve, [1, 2, 3, 5, 7, 10, 20, 30])

print(f"MXN Breakeven Inflation (TIIE nominal vs UDI real):")
print(f"{'Tenor':>6}  {'Nominal':>8}  {'Real':>8}  {'BEI':>8}")
print(f"{'─'*6}  {'─'*8}  {'─'*8}  {'─'*8}")
for row in bei:
    print(f"{row['years']:>4}Y  {row['nominal_rate']*100:>7.2f}%  "
          f"{row['real_rate']*100:>7.2f}%  {row['bei']*100:>7.2f}%")

# Plot
with apply_theme():
    fig, axes = create_figure(1)
    ax = axes[0]
    years = [r["years"] for r in bei]
    ax.plot(years, [r["nominal_rate"]*100 for r in bei], 'o-', label='Nominal (TIIE)', linewidth=2)
    ax.plot(years, [r["real_rate"]*100 for r in bei], 's-', label='Real (UDI)', linewidth=2)
    ax.fill_between(years,
                     [r["real_rate"]*100 for r in bei],
                     [r["nominal_rate"]*100 for r in bei],
                     alpha=0.2, label='Breakeven Inflation')
    ax.set_xlabel("Tenor (years)")
    ax.set_ylabel("Rate (%)")
    ax.set_title("Mexico — Nominal vs Real Rates & Breakeven Inflation")
    ax.legend()
    fig.tight_layout()

## Summary

| Instrument | Convention | Key Feature |
|---|---|---|
| TIIE Swap | 28-day periods, ACT/360 | Unique 28D structure |
| CETES | ACT/360, MXN 10 face | Discount bill |
| MBONO | Semi-annual, ACT/360 | Fixed-rate sovereign |
| Udibono | UDI-denominated, ACT/360 | Daily inflation linkage |
| BEI | Nominal - Real | ~6.5% implied inflation |